# Capítulo 6 — Funções: comandos reutilizáveis

## Missão
Transformar o código repetido em **funções** com argumentos, valores padrão, retorno de vários valores, `*args`, `**kwargs` e funções anônimas (`lambda`). O rover ganha uma biblioteca de cálculos de missão.

**No fim deste caderno você terá:** o módulo `gabarito_rover/calculos.py`, com estatísticas, consumo de bateria e conversões, todos documentados.

**Leitura essencial (opcional, livro em inglês):**
- Creating a Function · PDF p. 214
- Passing Information to a Function · PDF p. 216
- Returning Values from Functions · PDF p. 226
- Unmasking Anonymous Functions · PDF p. 227

**Gabarito:** a mesma trilha com o código pronto. Tente primeiro em `praticar/`.

## Preparação
Configuração do caderno (já vem pronta). Rode antes de tudo.

In [ ]:
import os, sys, json, time, math, random
sys.path.insert(0, "../comum")                 # dados.py (dados da missão do rover)
sys.path.insert(0, "..")                       # raiz: onde fica o seu pacote gabarito_rover/
os.makedirs("../gabarito_rover", exist_ok=True)
open("../gabarito_rover/__init__.py", "a").close()        # gabarito_rover/ é o rover que você constrói ao longo do livro
os.makedirs("../saidas", exist_ok=True)        # arquivos gerados nos exercícios
from dados import *
%load_ext autoreload
%autoreload 2
print("Pronto. Python", sys.version.split()[0], "| seu pacote: gabarito_rover")

## Ideia em 30 segundos
Uma função empacota um pedaço de lógica com um nome.
- A **docstring** explica o que ela faz.
- Valores padrão deixam argumentos opcionais.
- `return a, b` devolve uma tupla.
- `*args` recebe vários argumentos posicionais e `**kwargs` vários nomeados.
- `lambda` é uma função de uma linha, útil como `key=` em `sorted`, `max` e `min`.

### 6.1 **Primeira função com docstring.** Escreva `celsius_para_fahrenheit(c)` com docstring e teste com as temperaturas das 3 primeiras leituras. Mostre a docstring com `help`.

In [ ]:
def celsius_para_fahrenheit(c):
    """Converte graus Celsius para Fahrenheit."""
    return c * 9 / 5 + 32

print([round(celsius_para_fahrenheit(l[1]), 1) for l in LEITURAS[:3]])
help(celsius_para_fahrenheit)

### 6.2 **Padrões e argumentos nomeados.** Escreva `consumo(distancia_cm, terreno="plano", carga_g=0)`, que devolve o % de bateria gasto: 0,01% por cm, ×1,5 em terreno `"rochoso"` e + 0,001% por grama de carga. Chame com só a distância, com o terreno e com tudo por nome.

In [ ]:
def consumo(distancia_cm, terreno="plano", carga_g=0):
    """% de bateria gasto para andar distancia_cm."""
    fator = 1.5 if terreno == "rochoso" else 1.0
    return round(distancia_cm * 0.01 * fator + carga_g * 0.001, 2)

print(consumo(500), consumo(500, "rochoso"), consumo(carga_g=300, distancia_cm=500, terreno="rochoso"))

### 6.3 **Retornar vários valores.** Escreva `estatisticas(valores)`, que devolve mínimo, máximo e média. Use nas temperaturas e desempacote em 3 variáveis.

In [ ]:
def estatisticas(valores):
    """Devolve (mínimo, máximo, média)."""
    return min(valores), max(valores), sum(valores) / len(valores)

mn, mx, media = estatisticas([l[1] for l in LEITURAS])
print(mn, mx, round(media, 2))

### 6.4 **`*args` e `**kwargs`.** Escreva `registrar(evento, *detalhes, **dados)`, que monta uma linha como `[EVENTO] detalhe1 | detalhe2 | chave=valor`. Chame de 3 jeitos.

In [ ]:
def registrar(evento, *detalhes, **dados):
    partes = list(detalhes) + [f"{k}={v}" for k, v in dados.items()]
    return f"[{evento.upper()}] " + " | ".join(str(p) for p in partes)

print(registrar("inicio"))
print(registrar("coleta", "AM-001", "basalto"))
print(registrar("alerta", "rocha à frente", dist=18, acao="desviar"))

### 6.5 **Escopo.** Mostre a diferença entre uma variável local e uma global: uma função que tenta somar 1 a um contador global sem `global` (capture o `UnboundLocalError`) e outra que **retorna** o novo valor (a forma recomendada).

In [ ]:
contador = 0

def incrementa_errado():
    contador += 1

def incrementa(c):
    return c + 1

try:
    incrementa_errado()
except UnboundLocalError as e:
    print("erro:", e)
contador = incrementa(contador)
print(contador)

### 6.6 **lambda.** Com `max` e `key=lambda`, ache a leitura com a maior poeira. Ordene a `EQUIPE` pelo tamanho do nome e use `map` com uma lambda para gerar as baterias em fração.

In [ ]:
print(max(LEITURAS, key=lambda l: l[2])[0])
print([p["nome"] for p in sorted(EQUIPE, key=lambda p: len(p["nome"]))])
print(list(map(lambda l: l[3] / 100, LEITURAS))[:4])

### 6.7 **Módulo de cálculos.** Escreva `calculos.py` com `estatisticas`, `consumo` e `autonomia_cm(bateria, minima=30, terreno="plano")` (quantos cm o rover anda até chegar à bateria mínima). Cada função tem docstring e anotações de tipo.

_A célula grava `gabarito_rover/calculos.py`: a 1ª linha (`%%writefile`) já vem pronta, digite o código embaixo dela._

In [ ]:
%%writefile ../gabarito_rover/calculos.py
"""Cálculos da missão do rover."""


def estatisticas(valores: list[float]) -> tuple[float, float, float]:
    """Devolve (mínimo, máximo, média) de uma lista de números."""
    return min(valores), max(valores), sum(valores) / len(valores)


def consumo(distancia_cm: float, terreno: str = "plano", carga_g: float = 0) -> float:
    """% de bateria gasto para andar distancia_cm, conforme o terreno e a carga."""
    fator = 1.5 if terreno == "rochoso" else 1.0
    return round(distancia_cm * 0.01 * fator + carga_g * 0.001, 2)


def autonomia_cm(bateria: float, minima: float = 30, terreno: str = "plano") -> float:
    """Quantos cm o rover anda até chegar à bateria mínima."""
    fator = 1.5 if terreno == "rochoso" else 1.0
    return max(0.0, (bateria - minima) / (0.01 * fator))


### 6.8 **Teste.** Use `autonomia_cm` com a bateria da última leitura em terreno plano e rochoso.

In [ ]:
from gabarito_rover.calculos import autonomia_cm, estatisticas
bat = LEITURAS[-1][3]
print(autonomia_cm(bat), autonomia_cm(bat, terreno="rochoso"))
print(estatisticas([l[4] for l in LEITURAS]))

## Fechamento

- O que eu construí hoje:
- O que me surpreendeu:
- O que quero tentar sozinho a seguir:
